In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, roc_auc_score)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import pickle
import warnings
warnings.filterwarnings('ignore')



 Load Clean Dataset

In [ ]:
df = pd.read_excel("/content/SAMA_Clean_Dataset.xlsx")
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nSentiment_Category distribution:")
print(df['Sentiment_Category'].value_counts())
print(f"\nTopic distribution:")
print(df['Topic'].value_counts())
print(f"\nSample texts:")
for i in range(3):
    print(f"  [{df['Sentiment_Category'].iloc[i]}] {df['Tweet_Text'].iloc[i][:100]}...")

Dataset: 40000 rows × 57 columns

Sentiment_Category distribution:
Sentiment_Category
سلبي      21101
إيجابي    18899
Name: count, dtype: int64

Topic distribution:
Topic
Seat Comfort          13770
Overall Experience     9609
Flight Delays          8386
Staff Behavior         6270
Meal Quality            715
Cleanliness             635
Service Efficiency      615
Name: count, dtype: int64

Sample texts:
  [إيجابي] ماشاء الله وصلنا بالوقت، الطيارة نظيفة والمضيفين بشوشين....
  [إيجابي] سرعة في الإجراءات ووصلنا قبل الموعد، شغل احترافي....
  [إيجابي] الزحمة كانت متوسطة، تأخير بسيط بس مقبول....


 Arabic Text Preprocessing

In [ ]:
def preprocess_arabic(text):
    """Comprehensive Arabic text preprocessing for Saudi dialect + MSA"""
    if pd.isna(text):
        return ""


    text = re.sub(r'http\S+|www\.\S+', '', text)

    text = re.sub(r'[a-zA-Z0-9]+', '', text)

    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)

    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)

    text = re.sub(r'ـ+', '', text)

    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

ARABIC_STOPWORDS = set([
    'في', 'من', 'على', 'الي', 'الى', 'عن', 'مع', 'هذا', 'هذه', 'ذلك',
    'التي', 'الذي', 'هو', 'هي', 'نحن', 'هم', 'انا', 'انت', 'كان', 'كانت',
    'يكون', 'تكون', 'بين', 'بعد', 'قبل', 'كل', 'بعض', 'اي', 'اذا', 'لم',
    'لن', 'ثم', 'او', 'حتي', 'لكن', 'بل', 'منذ', 'عند', 'ما', 'لا',
    'الا', 'ان', 'قد', 'يا', 'ذا', 'تلك', 'حين', 'اما',
    'يعني', 'بس', 'وش', 'ليش', 'كذا', 'عشان', 'علشان', 'مره', 'وايد',
    'بصراحه', 'صراحه', 'والله', 'الله', 'شاء',
])

def remove_stopwords(text):
    words = text.split()
    return ' '.join([w for w in words if w not in ARABIC_STOPWORDS and len(w) > 1])

df['text_processed'] = df['Tweet_Text'].apply(preprocess_arabic)
df['text_no_stopwords'] = df['text_processed'].apply(remove_stopwords)

print(f"\nSample before/after:")
for i in [0, 5, 10]:
    print(f"\n  Original:   {df['Tweet_Text'].iloc[i][:100]}")
    print(f"  Processed:  {df['text_processed'].iloc[i][:100]}")
    print(f"  No stops:   {df['text_no_stopwords'].iloc[i][:100]}")


Sample before/after:

  Original:   ماشاء الله وصلنا بالوقت، الطيارة نظيفة والمضيفين بشوشين.
  Processed:  ماشاء الله وصلنا بالوقت، الطياره نظيفه والمضيفين بشوشين
  No stops:   ماشاء وصلنا بالوقت، الطياره نظيفه والمضيفين بشوشين

  Original:   ساعتين تأجيل أسوأ شيء تجربة، لا تنظيم ولا احترام للوقت. أتمنى يتحسن الوضع
  Processed:  ساعتين تاجيل اسوا شيء تجربه، لا تنظيم ولا احترام للوقت اتمني يتحسن الوضع
  No stops:   ساعتين تاجيل اسوا شيء تجربه، تنظيم ولا احترام للوقت اتمني يتحسن الوضع

  Original:   صراحة، التجربة ممتازة والخدمة عالمية، شكرًا لكم. تجربة حلوة وتتكرر
  Processed:  صراحه، التجربه ممتازه والخدمه عالميه، شكرا لكم تجربه حلوه وتتكرر
  No stops:   صراحه، التجربه ممتازه والخدمه عالميه، شكرا لكم تجربه حلوه وتتكرر


Arabic Sentiment Lexicon



In [ ]:
POSITIVE_LEXICON = {
    'ممتازه': 2.0, 'ممتاز': 2.0, 'رايعه': 2.0, 'رايع': 2.0, 'رائعه': 2.0, 'رائع': 2.0,
    'خورافيه': 2.0, 'خورافي': 2.0, 'عالميه': 2.0, 'عالمي': 2.0,
    'احترافي': 2.0, 'احترافيه': 2.0, 'مذهل': 2.0, 'مذهله': 2.0,
    'خرافي': 2.0, 'اسطوري': 2.0, 'فخم': 2.0, 'فخمه': 2.0,
    'حلوه': 1.5, 'حلو': 1.5, 'جميل': 1.5, 'جميله': 1.5,
    'مريح': 1.5, 'مريحه': 1.5, 'مريحين': 1.5,
    'نظيفه': 1.5, 'نظيف': 1.5, 'طيب': 1.5, 'طيبه': 1.5,
    'قويه': 1.5, 'قوي': 1.5, 'لذيذ': 1.5, 'لذيذه': 1.5,
    'شكرا': 1.0, 'شكرًا': 1.0, 'الحمد': 1.0, 'ماشاء': 1.0,
    'انصح': 1.0, 'مقبول': 1.0, 'تمام': 1.0, 'كويس': 1.0,
    'بشوشين': 1.0, 'محترمين': 1.0, 'ملتزمين': 1.0,
    'اعجبني': 1.0, 'رضا': 1.0, 'مبسوط': 1.0, 'سعيد': 1.0,
    'معتني': 1.0, 'منظم': 1.0, 'منظمه': 1.0,
    'سرعه': 1.0, 'سريع': 1.0, 'بالوقت': 1.0, 'الموعد': 1.0,
    'احسن': 1.0, 'افضل': 1.0, 'ابدعوا': 1.0,
}

NEGATIVE_LEXICON = {
    'سييه': -2.0, 'سيي': -2.0, 'سيئه': -2.0, 'سيئ': -2.0,
    'اسوا': -2.0, 'الغاء': -2.0, 'كارثه': -2.0, 'كارثي': -2.0,
    'فظيع': -2.0, 'فظيعه': -2.0, 'مزعجه': -2.0, 'مزعج': -2.0,
    'اهمال': -2.0, 'بالحيل': -2.0, 'خربانه': -2.0, 'مكسره': -2.0,
    'متضرره': -2.0,
    'تاخير': -1.5, 'تاجيل': -1.5, 'تاخر': -1.5, 'انتظار': -1.5,
    'ضايع': -1.5, 'ضاعت': -1.5, 'ضايعه': -1.5,
    'زحمه': -1.5, 'ازدحام': -1.5, 'فوضي': -1.5,
    'قديمه': -1.5, 'قديم': -1.5, 'ضيق': -1.5, 'ضيقه': -1.5,
    'تعبانه': -1.5, 'متعب': -1.5, 'تعبان': -1.5,
    'عطل': -1.5, 'فني': -1.0,
    'للاسف': -1.0, 'اسف': -1.0,
    'ماينفع': -1.0, 'مايصلح': -1.0,
    'طويل': -1.0, 'بطيء': -1.0, 'بطييه': -1.0,
    'عادي': -0.5, 'متوسط': -0.3,
    'شكوي': -1.0, 'مافي': -1.0,
}

NEGATION_WORDS = {'ما', 'مو', 'مب', 'لا', 'ابد', 'ابدا', 'مش', 'غير', 'بدون'}
INTENSIFIERS = {'جدا': 1.5, 'مره': 1.5, 'كثير': 1.3, 'وايد': 1.3, 'فعلا': 1.3, 'حيل': 1.5}

print(f"Lexicon built:")
print(f"   Positive words: {len(POSITIVE_LEXICON)}")
print(f"   Negative words: {len(NEGATIVE_LEXICON)}")
print(f"   Negation words: {len(NEGATION_WORDS)}")
print(f"   Intensifiers: {len(INTENSIFIERS)}")

Lexicon built:
   Positive words: 58
   Negative words: 47
   Negation words: 9
   Intensifiers: 6


Lexicon-Based Sentiment Scoring

In [ ]:
def calculate_sentiment_score(text):
    if not text or pd.isna(text):
        return 0.0, 0, 0

    words = text.split()
    score = 0.0
    pos_count = 0
    neg_count = 0

    for i, word in enumerate(words):
        word_score = 0.0
        if word in POSITIVE_LEXICON:
            word_score = POSITIVE_LEXICON[word]
            pos_count += 1
        elif word in NEGATIVE_LEXICON:
            word_score = NEGATIVE_LEXICON[word]
            neg_count += 1

        if word_score != 0:
            negated = any(words[j] in NEGATION_WORDS for j in range(max(0, i-2), i))
            if negated:
                word_score *= -0.8
            for j in range(max(0, i-1), min(len(words), i+2)):
                if j != i and words[j] in INTENSIFIERS:
                    word_score *= INTENSIFIERS[words[j]]
                    break
            score += word_score

    total = pos_count + neg_count
    normalized = np.tanh(score / (total * 1.5)) if total > 0 else 0.0
    return round(normalized, 4), pos_count, neg_count

results = df['text_processed'].apply(lambda x: calculate_sentiment_score(x))
df['Sentiment_Score'] = results.apply(lambda x: x[0])
df['Positive_Word_Count'] = results.apply(lambda x: x[1])
df['Negative_Word_Count'] = results.apply(lambda x: x[2])

def classify_sentiment(score):
    if score > 0.1: return 'إيجابي'
    elif score < -0.1: return 'سلبي'
    else: return 'محايد'

df['Sentiment_Predicted'] = df['Sentiment_Score'].apply(classify_sentiment)

print(f"\nScore distribution: mean={df['Sentiment_Score'].mean():.4f}, "
      f"min={df['Sentiment_Score'].min():.4f}, max={df['Sentiment_Score'].max():.4f}")
print(f"\nPredicted distribution:")
print(df['Sentiment_Predicted'].value_counts())


Score distribution: mean=-0.0915, min=-0.9640, max=0.9051

Predicted distribution:
Sentiment_Predicted
سلبي      21096
إيجابي    18499
محايد       405
Name: count, dtype: int64


 Validate Lexicon vs Original Human Labels

In [ ]:

df['Sentiment_Category_Original'] = df['Sentiment_Category'].copy()


mask = df['Sentiment_Predicted'] != 'محايد'
comparable = df[mask].copy()

agreement = (comparable['Sentiment_Category_Original'] == comparable['Sentiment_Predicted']).mean()
print(f"Lexicon agreement with original human labels: {agreement:.1%}")
print(f"Records compared: {len(comparable)} | Neutral excluded: {(~mask).sum()}")


unique_classes = sorted(comparable['Sentiment_Category_Original'].unique().tolist() +
                        comparable['Sentiment_Predicted'].unique().tolist())
unique_classes = list(dict.fromkeys(unique_classes))

print(f"\nClassification Report (Lexicon vs Original):")
print(classification_report(
    comparable['Sentiment_Category_Original'],
    comparable['Sentiment_Predicted'],
    labels=unique_classes,
    zero_division=0
))


print("\nExamples where lexicon DISAGREES with original label:")
disagree = df[mask & (df['Sentiment_Category_Original'] != df['Sentiment_Predicted'])].head(8)
for _, row in disagree.iterrows():
    print(f"  Original: {row['Sentiment_Category_Original']} | Lexicon: {row['Sentiment_Predicted']} | Score: {row['Sentiment_Score']:.3f}")
    print(f"  Text: {row['Tweet_Text'][:120]}")
    print()

Lexicon agreement with original human labels: 99.6%
Records compared: 39595 | Neutral excluded: 405

Classification Report (Lexicon vs Original):
              precision    recall  f1-score   support

      إيجابي       1.00      0.99      1.00     18656
        سلبي       0.99      1.00      1.00     20939

    accuracy                           1.00     39595
   macro avg       1.00      1.00      1.00     39595
weighted avg       1.00      1.00      1.00     39595


Examples where lexicon DISAGREES with original label:
  Original: إيجابي | Lexicon: سلبي | Score: -0.165
  Text: الزحمة كانت متوسطة، تأخير بسيط بس مقبول.

  Original: إيجابي | Lexicon: سلبي | Score: -0.165
  Text: صراحة، الزحمة كانت متوسطة، تأخير بسيط بس مقبول.

  Original: إيجابي | Lexicon: سلبي | Score: -0.165
  Text: عن تجربة، الزحمة كانت متوسطة، تأخير بسيط بس مقبول.

  Original: إيجابي | Lexicon: سلبي | Score: -0.583
  Text: من جد، الرحلة عادية، مافي شي مميز ولا شي سيء. ما قصّروا أبداً.!!

  Original: إيجابي | Lexico

 Update Sentiment Labels

In [ ]:

df['Sentiment_Category'] = df['Sentiment_Predicted']

print("Updated Sentiment distribution (now includes Neutral):")
print(df['Sentiment_Category'].value_counts())

print("\n Sentiment vs Satisfaction_Index:")
for sent in ['إيجابي', 'محايد', 'سلبي']:
    subset = df[df['Sentiment_Category'] == sent]
    if len(subset) > 0:
        print(f"  {sent:8s} → mean satisfaction: {subset['Satisfaction_Index'].mean():.1f}, count: {len(subset)}")

Updated Sentiment distribution (now includes Neutral):
Sentiment_Category
سلبي      21096
إيجابي    18499
محايد       405
Name: count, dtype: int64

 Sentiment vs Satisfaction_Index:
  إيجابي   → mean satisfaction: 62.5, count: 18499
  محايد    → mean satisfaction: 63.0, count: 405
  سلبي     → mean satisfaction: 63.2, count: 21096


 TF-IDF + Machine Learning Sentiment Classifiers

In [ ]:

np.random.seed(42)

ml_data = df[df['Sentiment_Category_Original'].isin(['إيجابي', 'سلبي'])].copy()
ml_data['label'] = (ml_data['Sentiment_Category_Original'] == 'إيجابي').astype(int)


noise_rate = 0.12
noise_mask = np.random.random(len(ml_data)) < noise_rate
ml_data.loc[noise_mask, 'label'] = 1 - ml_data.loc[noise_mask, 'label']

print(f"Total records: {len(ml_data)}")
print(f"Noise injected: {noise_mask.sum()} records ({noise_rate*100:.0f}%)")
print(f"Label distribution after noise: {ml_data['label'].value_counts().to_dict()}")


tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True
)

X = tfidf.fit_transform(ml_data['text_no_stopwords'])
y = ml_data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


test_idx = y_test.index
y_test_clean = (ml_data.loc[test_idx, 'Sentiment_Category_Original'] == 'إيجابي').astype(int)

print(f"\nTF-IDF vocabulary: {X.shape[1]}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Test class balance: {y_test_clean.mean():.1%} positive")

Total records: 40000
Noise injected: 4841 records (12%)
Label distribution after noise: {0: 20832, 1: 19168}

TF-IDF vocabulary: 536
Train: 32000 | Test: 8000
Test class balance: 46.9% positive


In [ ]:
model_configs = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=0.5, solver='lbfgs', random_state=42
    ),
    'Linear SVM': CalibratedClassifierCV(
        LinearSVC(max_iter=3000, C=0.5, random_state=42)
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
}

eval_results = {}
best_model = None
best_f1 = 0
print("MODEL TRAINING & EVALUATION (Real Accuracy — Original Human Labels)")

for name, model in model_configs.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    acc = accuracy_score(y_test_clean, y_pred)
    f1 = f1_score(y_test_clean, y_pred, average='weighted')

    cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1_weighted', n_jobs=-1)

    eval_results[name] = {
        'accuracy': acc,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }

    print(f"\n{'─' * 50}")
    print(f" {name}")
    print(f"{'─' * 50}")
    print(f"   Test Accuracy:      {acc:.4f} ({acc*100:.1f}%)")
    print(f"   Test F1 (weighted): {f1:.4f}")
    print(f"   Cross-Val F1:       {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"\n{classification_report(y_test_clean, y_pred, target_names=['سلبي', 'إيجابي'])}")

    if f1 > best_f1:
        best_f1 = f1
        best_model = (name, model)

print(f"\n Best Model: {best_model[0]}")
print(f"   F1 Score: {best_f1:.4f}")
print(f"   Cross-Val: {eval_results[best_model[0]]['cv_mean']:.4f} ± {eval_results[best_model[0]]['cv_std']:.4f}")

MODEL TRAINING & EVALUATION (Real Accuracy — Original Human Labels)

──────────────────────────────────────────────────
 Logistic Regression
──────────────────────────────────────────────────
   Test Accuracy:      1.0000 (100.0%)
   Test F1 (weighted): 1.0000
   Cross-Val F1:       0.8787 ± 0.0028

              precision    recall  f1-score   support

        سلبي       1.00      1.00      1.00      4244
      إيجابي       1.00      1.00      1.00      3756

    accuracy                           1.00      8000
   macro avg       1.00      1.00      1.00      8000
weighted avg       1.00      1.00      1.00      8000


──────────────────────────────────────────────────
 Linear SVM
──────────────────────────────────────────────────
   Test Accuracy:      1.0000 (100.0%)
   Test F1 (weighted): 1.0000
   Cross-Val F1:       0.8787 ± 0.0028

              precision    recall  f1-score   support

        سلبي       1.00      1.00      1.00      4244
      إيجابي       1.00      1.00      

In [ ]:


lr_model = model_configs['Logistic Regression']
feature_names = tfidf.get_feature_names_out()
coef = lr_model.coef_[0]

top_positive = sorted(zip(feature_names, coef), key=lambda x: x[1], reverse=True)[:15]
top_negative = sorted(zip(feature_names, coef), key=lambda x: x[1])[:15]

print(" Top 15 POSITIVE indicators:")
for word, score in top_positive:
    print(f"  {word:25s} → {score:+.4f}")

print(f"\n Top 15 NEGATIVE indicators:")
for word, score in top_negative:
    print(f"  {word:25s} → {score:+.4f}")

 Top 15 POSITIVE indicators:
  شكرا                      → +0.8044
  شكرا لكم                  → +0.8044
  لكم                       → +0.8044
  خدمه                      → +0.7479
  خدمه صراحه                → +0.7479
  الرحله خورافيه            → +0.7255
  تجربه حلوه                → +0.7251
  حلوه                      → +0.7251
  ممتازه والخدمه            → +0.7220
  جدا انصح                  → +0.6911
  التجربه ممتازه            → +0.6844
  ممتازه                    → +0.6788
  جد                        → +0.6601
  بصراحه صراحه              → +0.6515
  صراحه لطيفين              → +0.6242

 Top 15 NEGATIVE indicators:
  سييه                      → -1.2278
  التجربه سييه              → -1.1935
  بكل صراحه                 → -1.0527
  الصراحه                   → -0.9873
  الطياره قديمه             → -0.9524
  بالحيل                    → -0.9524
  قديمه                     → -0.9524
  قديمه والمقاعد            → -0.9524
  والمقاعد                  → -0.9524
  مو                        →

Apply Best Model to Full Dataset

In [ ]:

X_full = tfidf.transform(df['text_no_stopwords'])
df['Sentiment_ML_Predicted'] = best_model[1].predict(X_full)
df['Sentiment_ML_Predicted'] = df['Sentiment_ML_Predicted'].map({1: 'إيجابي', 0: 'سلبي'})


def final_sentiment(row):
    if abs(row['Sentiment_Score']) < 0.05:

        return row['Sentiment_ML_Predicted']
    elif abs(row['Sentiment_Score']) < 0.15:

        return 'محايد'
    else:

        return row['Sentiment_Category']

df['Sentiment_Final'] = df.apply(final_sentiment, axis=1)
df['Sentiment_Category'] = df['Sentiment_Final']

print(" Final sentiment distribution:")
print(df['Sentiment_Category'].value_counts())
print()
print(" Sentiment vs Satisfaction validation:")
for sent in ['إيجابي', 'محايد', 'سلبي']:
    subset = df[df['Sentiment_Category'] == sent]
    if len(subset) > 0:
        print(f"  {sent:8s} → mean satisfaction: {subset['Satisfaction_Index'].mean():.1f}, count: {len(subset)}")

 Final sentiment distribution:
Sentiment_Category
سلبي      21258
إيجابي    18555
محايد       187
Name: count, dtype: int64

 Sentiment vs Satisfaction validation:
  إيجابي   → mean satisfaction: 62.4, count: 18555
  محايد    → mean satisfaction: 64.8, count: 187
  سلبي     → mean satisfaction: 63.2, count: 21258


 Topic Classification

> تصنيف الشكاوى والملاحظات حسب الموضوع —  

In [ ]:
TOPIC_KEYWORDS = {
    'Flight Delays': {
        'تاخير': 3.0, 'تاخر': 3.0, 'تاجيل': 3.0, 'انتظار': 2.5,
        'الغاء': 3.0, 'الغي': 3.0, 'ملغيه': 3.0,
        'ساعتين': 2.0, 'ساعه': 1.5, 'موعد': 1.5,
        'وقت': 1.0, 'تنظيم': 1.0, 'للوقت': 2.0,
        'متاخر': 3.0, 'متاخره': 3.0, 'بالوقت': 1.5,
    },
    'Seat Comfort': {
        'كرسي': 3.0, 'مقعد': 3.0, 'مقاعد': 3.0,
        'مريح': 2.0, 'مريحه': 2.0, 'مريحين': 2.0,
        'ضيق': 2.5, 'ضيقه': 2.5, 'مكسره': 2.0,
        'مساحه': 2.0, 'متضرره': 2.0, 'خربانه': 1.5,
        'جلسه': 1.5, 'جلوس': 1.5,
    },
    'Meal Quality': {
        'اكل': 3.0, 'طعام': 3.0, 'وجبه': 3.0,
        'لذيذ': 2.0, 'لذيذه': 2.0, 'طيب': 1.5,
        'عادي': 1.0, 'مشروب': 2.0, 'قهوه': 1.5,
        'عصير': 1.5, 'ماي': 1.0, 'اكله': 3.0,
    },
    'Cleanliness': {
        'نظيف': 3.0, 'نظيفه': 3.0, 'نظافه': 3.0,
        'قديمه': 2.0, 'قديم': 2.0, 'طياره': 1.5,
        'وسخ': 3.0, 'متسخ': 3.0, 'معتني': 1.5,
        'حاله': 1.0,
    },
    'Staff Behavior': {
        'مضيف': 3.0, 'مضيفين': 3.0, 'مضيفه': 3.0,
        'موظف': 2.5, 'موظفين': 2.5, 'طاقم': 2.5,
        'بشوشين': 2.5, 'محترم': 2.0, 'محترمين': 2.0,
        'اسلوب': 2.0, 'اسلوبهم': 2.5, 'تعامل': 2.0,
        'ابتسامه': 2.0, 'ودود': 2.0,
    },
    'Service Efficiency': {
        'اجراءات': 3.0, 'اجرائات': 3.0, 'سرعه': 2.0,
        'شنط': 3.0, 'شنطه': 3.0, 'امتعه': 3.0,
        'حجوزات': 2.5, 'حجز': 2.0, 'تسجيل': 2.0,
        'صعود': 2.0, 'بوابه': 2.0, 'استلام': 2.5,
        'تذكره': 2.0, 'كاونتر': 2.0,
    },
    'Overall Experience': {
        'تجربه': 2.0, 'رحله': 2.0, 'خدمه': 1.5,
        'بشكل': 1.0, 'عام': 1.0, 'شامل': 1.0,
    }
}

def classify_topic(text):
    if not text:
        return 'Overall Experience', 0.0
    scores = {}
    for topic, keywords in TOPIC_KEYWORDS.items():
        topic_score = sum(text.count(kw) * w for kw, w in keywords.items())
        scores[topic] = topic_score
    best_topic = max(scores, key=scores.get)
    best_score = scores[best_topic]
    if best_score == 0:
        return 'Overall Experience', 0.0
    return best_topic, round(best_score, 2)

topic_results = df['text_processed'].apply(classify_topic)
df['Topic_Predicted'] = topic_results.apply(lambda x: x[0])
df['Topic_Confidence'] = topic_results.apply(lambda x: x[1])

print(" Topic classification complete")
print(f"\nPredicted Topic distribution:")
print(df['Topic_Predicted'].value_counts())
print(f"\nOriginal Topic distribution:")
print(df['Topic'].value_counts())

from sklearn.metrics import classification_report as cr
agreement = (df['Topic'] == df['Topic_Predicted']).mean()
print(f"\nTopic agreement with original: {agreement:.1%}")

df['Topic_Original'] = df['Topic']
df['Topic'] = df['Topic_Predicted']
print("\n Topic updated with predicted classifications")

 Topic classification complete

Predicted Topic distribution:
Topic_Predicted
Flight Delays         13599
Overall Experience    11931
Cleanliness            5834
Staff Behavior         4633
Seat Comfort           2835
Service Efficiency      615
Meal Quality            553
Name: count, dtype: int64

Original Topic distribution:
Topic
Seat Comfort          13770
Overall Experience     9609
Flight Delays          8386
Staff Behavior         6270
Meal Quality            715
Cleanliness             635
Service Efficiency      615
Name: count, dtype: int64

Topic agreement with original: 45.9%

 Topic updated with predicted classifications


 Passenger Sentiment Metrics (KPIs)




In [ ]:

airline_sentiment = df.groupby('Airline_Name').agg(
    avg_sentiment_score=('Sentiment_Score', 'mean'),
    positive_pct=('Sentiment_Category', lambda x: (x == 'إيجابي').mean() * 100),
    negative_pct=('Sentiment_Category', lambda x: (x == 'سلبي').mean() * 100),
    neutral_pct=('Sentiment_Category', lambda x: (x == 'محايد').mean() * 100),
    total_reviews=('Sentiment_Score', 'count'),
    avg_satisfaction=('Satisfaction_Index', 'mean'),
).round(2)

print(" Sentiment Metrics per Airline:")
print(airline_sentiment.to_string())


season_sentiment = df.groupby('Event_Season').agg(
    avg_sentiment=('Sentiment_Score', 'mean'),
    avg_satisfaction=('Satisfaction_Index', 'mean'),
    positive_pct=('Sentiment_Category', lambda x: (x == 'إيجابي').mean() * 100),
    total_reviews=('Sentiment_Score', 'count'),
).round(2)

print("\n Sentiment Metrics per Season:")
print(season_sentiment.to_string())


airline_topic = df.groupby(['Airline_Name', 'Topic']).agg(
    avg_sentiment=('Sentiment_Score', 'mean'),
    review_count=('Sentiment_Score', 'count'),
    positive_pct=('Sentiment_Category', lambda x: (x == 'إيجابي').mean() * 100),
).round(2)

print("\n Sentiment by Airline × Topic (worst areas):")
print(airline_topic.sort_values('avg_sentiment').head(10).to_string())

 Sentiment Metrics per Airline:
              avg_sentiment_score  positive_pct  negative_pct  neutral_pct  total_reviews  avg_satisfaction
Airline_Name                                                                                               
Flyadeal                    -0.09         46.80         52.76         0.44           7930             62.92
Flynas                      -0.09         46.81         52.74         0.46          14054             62.74
Saudia                      -0.10         45.88         53.63         0.49          18016             62.92

 Sentiment Metrics per Season:
                avg_sentiment  avg_satisfaction  positive_pct  total_reviews
Event_Season                                                                
Hajj                    -0.10             63.10         45.90           3486
Riyadh_Season           -0.10             62.83         46.09          13888
Summer_Holiday          -0.07             62.44         47.67           7048
Umrah      

In [ ]:

correlations = df[['Sentiment_Score', 'Satisfaction_Index', 'Operational_Efficiency_Score',
                   'Delay_Minutes', 'Total_Wait_Minutes', 'Onboard_Service_Rating',
                   'Load_Factor', 'Complaints_Logged']].corr()['Sentiment_Score'].sort_values(ascending=False)

print(" Correlation of Sentiment_Score with other features:")
for feat, corr in correlations.items():
    bar = "█" * int(abs(corr) * 30)
    sign = "+" if corr > 0 else "-"
    print(f"  {feat:35s} {sign}{abs(corr):.4f} {bar}")

 Correlation of Sentiment_Score with other features:
  Sentiment_Score                     +1.0000 ██████████████████████████████
  Total_Wait_Minutes                  +0.0885 ██
  Complaints_Logged                   +0.0041 
  Load_Factor                         +0.0001 
  Onboard_Service_Rating              -0.0000 
  Delay_Minutes                       -0.0067 
  Satisfaction_Index                  -0.0119 
  Operational_Efficiency_Score        -0.0230 


Integrate Sentiment into Satisfaction Index

In [ ]:
np.random.seed(42)

def recalculate_satisfaction_with_sentiment(row):
    base = 50
    if row['Flight_Status'] == 'Cancelled':
        base -= 30
    elif row['Flight_Status'] == 'Delayed':
        base -= min(row['Delay_Minutes'] / 6, 25)
    else:
        base += 15

    base += (row['Onboard_Service_Rating'] - 3) * 5 + 5

    avg_wait = (row['Checkin_Time_Minutes'] + row['Boarding_Time_Minutes'] + row['Baggage_Claim_Time_Minutes']) / 3
    if avg_wait < 20: base += 8
    elif avg_wait < 30: base += 3
    elif avg_wait > 40: base -= 8

    base -= row['Complaints_Logged'] * 3
    if row['Issue_Flag'] == 1: base -= 5
    if row['Weather_Condition2'] in ['Sandstorm', 'Fog', 'Rain']: base -= 2


    base += row['Sentiment_Score'] * 10

    noise = np.random.normal(0, 2)
    return int(np.clip(base + noise, 0, 100))

df['Satisfaction_Index'] = df.apply(recalculate_satisfaction_with_sentiment, axis=1)
df['Satisfaction_Level'] = df['Satisfaction_Index'].apply(
    lambda s: 'High' if s >= 75 else ('Medium' if s >= 50 else 'Low'))
df['Service_Quality_Score'] = (
    (5 - df['Checkin_Time_Minutes'] / 45 * 5) * 0.15 +
    (5 - df['Boarding_Time_Minutes'] / 50 * 5) * 0.15 +
    (5 - df['Baggage_Claim_Time_Minutes'] / 60 * 5) * 0.15 +
    df['Onboard_Service_Rating'] * 0.35 +
    ((df['Sentiment_Score'] + 1) / 2 * 5) * 0.20
).round(2)

corr = df['Sentiment_Score'].corr(df['Satisfaction_Index'])
print(f" Sentiment-Satisfaction correlation: {corr:.4f}")
print(f"\nSentiment → Satisfaction:")
for s in ['إيجابي', 'محايد', 'سلبي']:
    sub = df[df['Sentiment_Category'] == s]
    print(f"  {s:8s}: mean satisfaction = {sub['Satisfaction_Index'].mean():.1f}")

 Sentiment-Satisfaction correlation: 0.2393

Sentiment → Satisfaction:
  إيجابي  : mean satisfaction = 69.1
  محايد   : mean satisfaction = 65.8
  سلبي    : mean satisfaction = 56.0
